https://aclanthology.org/2025.naacl-long.57.pdf

https://app.notion.com/p/Unifying-AI-Tutor-Evaluation-An-Evaluation-Taxonomy-for-Pedagogical-Ability-Assessment-of-LLM-Power-371b561b6ea38024831ecf85d67d7c2a?source=copy_link

In [3]:
import asyncio
import os
from pathlib import Path

import httpx
import pandas as pd
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field, create_model

load_dotenv(".env")
if not os.getenv("OPENAI_API_KEY"):
    alias = os.getenv("OPEN_API_KEY")
    if not alias:
        raise RuntimeError("Set OPENAI_API_KEY (or OPEN_API_KEY) in .env")
    os.environ["OPENAI_API_KEY"] = alias

YN = ["Yes", "To some extent", "No"]
TAXONOMY = {
    "mistake_identification": {"definition": "Has the tutor identified/recognized a mistake in the student's response?", "labels": YN, "desired": "Yes"},
    "mistake_location": {"definition": "Does the tutor's response accurately point to a genuine mistake and its location?", "labels": YN, "desired": "Yes"},
    "revealing_answer": {"definition": "Does the tutor reveal the final answer (whether correct or not)?", "labels": ["Yes (correct)", "Yes (incorrect)", "No"], "desired": "No"},
    "providing_guidance": {
        "definition": "Does the tutor offer correct and relevant guidance, such as an explanation, elaboration, hint, examples, etc.?",
        "labels": YN, "desired": "Yes",
        "note": "Yes = guidance is correct and relevant. To some extent = guidance exists but is incorrect or incomplete. No = no guidance.",
    },
    "actionability": {"definition": "Is it clear from the tutor's feedback what the student should do next?", "labels": YN, "desired": "Yes"},
    "coherence": {"definition": "Is the tutor response logically consistent with the student's previous responses?", "labels": YN, "desired": "Yes"},
    "tutor_tone": {"definition": "Is the tutor's response encouraging, neutral, or offensive?", "labels": ["Encouraging", "Neutral", "Offensive"], "desired": "Encouraging"},
    "human_likeness": {"definition": "Does the tutor's response sound natural rather than robotic or artificial?", "labels": YN, "desired": "Yes"},
}
LABEL_TO_SCORE = {dim: {lab: i + 1 for i, lab in enumerate(spec["labels"])} for dim, spec in TAXONOMY.items()}


def taxonomy_text() -> str:
    blocks = []
    for dim, spec in TAXONOMY.items():
        note = f"\n  Note: {spec['note']}" if "note" in spec else ""
        blocks.append(
            f"- {dim}\n  Definition: {spec['definition']}\n"
            f"  Allowed labels: {spec['labels']}\n  Desired label: {spec['desired']}{note}"
        )
    return "\n".join(blocks)


class DimensionEval(BaseModel):
    label: str = Field(description="One allowed label for this dimension")
    reasoning: str = Field(description="One-sentence justification")


PedagogyEvaluation = create_model(
    "PedagogyEvaluation",
    **{dim: (DimensionEval, ...) for dim in TAXONOMY},
)

PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a strict and objective evaluator of tutor responses for student mistake/confusion remediation.
Evaluate the tutor response independently on exactly the eight taxonomy dimensions below.

[TAXONOMY]
{taxonomy}

[RULES]
1. Judge every dimension independently.
2. Do not propagate a failure in one dimension to another.
3. For Providing Guidance, evaluate correctness and relevance to the student's actual mistake.
4. For Revealing Answer, determine whether the final answer was revealed and, if so, whether it was correct.
5. For Actionability, determine whether the student clearly knows what to do next.
6. For Coherence, judge logical consistency with the preceding student response(s).
7. Use "To some extent" when evidence is genuinely partial or ambiguous and that label is allowed.
8. Never invent new labels.
9. Keep each reasoning field to one sentence."""),
    ("human", """[NOW EVALUATE]
Problem: {problem}
Student's response: {student_response}
Optional reference solution: {reference_solution}
Tutor response: {tutor_response}"""),
])

MODEL = "gpt-4.1-mini"
HTTP = dict(limits=httpx.Limits(max_connections=256, max_keepalive_connections=64), timeout=httpx.Timeout(120.0))
llm = ChatOpenAI(
    model=MODEL, temperature=0, timeout=120, max_retries=6,
    http_client=httpx.Client(**HTTP),
    http_async_client=httpx.AsyncClient(**HTTP),
)
chain = PROMPT | llm.with_structured_output(PedagogyEvaluation)


def validate_and_enrich(result) -> dict:
    payload = result.model_dump() if hasattr(result, "model_dump") else result
    out = {}
    for dim, spec in TAXONOMY.items():
        label = payload[dim]["label"]
        if label not in spec["labels"]:
            raise ValueError(f"{dim}: invalid label {label!r}; allowed={spec['labels']}")
        out[dim] = {
            "label": label,
            "score": LABEL_TO_SCORE[dim][label],
            "desired_match": label == spec["desired"],
            "reasoning": str(payload[dim].get("reasoning", "")).strip(),
        }
    return out


def _payload(ex, taxonomy):
    return {
        "taxonomy": taxonomy,
        "problem": ex["problem"],
        "student_response": ex["student_response"],
        "reference_solution": ex.get("reference_solution") or "Not provided",
        "tutor_response": ex["tutor_response"],
    }


def evaluate_one(problem, student_response, tutor_response, reference_solution=None):
    return validate_and_enrich(chain.invoke(_payload({
        "problem": problem,
        "student_response": student_response,
        "tutor_response": tutor_response,
        "reference_solution": reference_solution,
    }, taxonomy_text())))


async def evaluate_many(examples, max_concurrency=10):
    taxonomy = taxonomy_text()
    total = len(examples)
    results, errors = [], []
    if total == 0:
        return results, errors

    sem = asyncio.Semaphore(max_concurrency)
    every = max(1, min(25, max_concurrency))
    print(f"start: 0/{total} (max_concurrency={max_concurrency})", flush=True)

    async def run_one(ex):
        async with sem:
            try:
                return ex, await chain.ainvoke(_payload(ex, taxonomy)), None
            except Exception as e:
                return ex, None, e

    done = 0
    for fut in asyncio.as_completed([asyncio.create_task(run_one(ex)) for ex in examples]):
        ex, parsed, err = await fut
        done += 1
        llm_name = ex.get("llm_name", "")
        try:
            if err:
                raise err
            results.append({
                "idx": ex["idx"], "llm_name": ex["llm_name"],
                "error_category": ex.get("error_category"),
                "tutor_response": ex["tutor_response"],
                "evaluation": validate_and_enrich(parsed),
            })
        except Exception as e:
            errors.append({"idx": ex["idx"], "llm_name": llm_name, "error": str(e)})
            print(f"{llm_name} row {ex['idx']} failed: {e}", flush=True)
        if done == total or done % every == 0:
            print(f"progress: {done}/{total} (ok {len(results)} / fail {len(errors)})", flush=True)

    results.sort(key=lambda x: (str(x.get("llm_name") or ""), x["idx"]))
    return results, errors


def results_to_dataframe(results):
    rows = []
    for item in results:
        row = {k: item.get(k) for k in ("idx", "llm_name", "error_category", "tutor_response")}
        for dim, ev in item["evaluation"].items():
            row[f"{dim}_label"] = ev["label"]
            row[f"{dim}_score"] = ev["score"]
            row[f"{dim}_desired"] = ev["desired_match"]
            row[f"{dim}_reasoning"] = ev["reasoning"]
        rows.append(row)
    return pd.DataFrame(rows)


def _groups(df, group_cols):
    group_cols = list(group_cols or [])
    if not group_cols:
        yield {}, df
        return
    for key, sub in df.groupby(group_cols, dropna=False, sort=False):
        key_t = key if isinstance(key, tuple) else (key,)
        yield dict(zip(group_cols, key_t)), sub


def desired_match_rates(df, group_cols=None):
    rows = []
    for meta, sub in _groups(df, group_cols):
        for dim, spec in TAXONOMY.items():
            rows.append({**meta, "n": int(len(sub)), "dimension": dim, "desired_label": spec["desired"], "DAMR": float(sub[f"{dim}_desired"].mean())})
    return pd.DataFrame(rows)


def label_distribution(df, group_cols=None):
    rows = []
    for meta, sub in _groups(df, group_cols):
        for dim in TAXONOMY:
            counts = sub[f"{dim}_label"].value_counts(dropna=False)
            total = counts.sum()
            rows.extend(
                {**meta, "n": int(len(sub)), "dimension": dim, "label": label, "count": int(n), "proportion": float(n / total)}
                for label, n in counts.items()
            )
    return pd.DataFrame(rows)


In [4]:
DATA_PATH = "Final__Inference_Generation_SFT_DPO__sorted_df.csv"
SAMPLE_N = None  # None = full CSV
LLM_COLS = ["base", "sft", "dpo_01", "dpo_03", "dpo_05"]
MAX_CONCURRENCY = 200
OUT_DIR = Path("pedagogy_evaluation")
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
eval_df = df if SAMPLE_N is None else df.head(SAMPLE_N).copy()
examples = [
    {
        "idx": i,
        "llm_name": llm,
        "problem": row["problem"],
        "student_response": row["student_mistake"],
        "error_category": row["error_category"],
        "tutor_response": row[llm],
        "reference_solution": row.get("reference_solution"),
    }
    for llm in LLM_COLS
    for i, row in eval_df.iterrows()
]
print(f"{len(eval_df)} rows × {len(LLM_COLS)} models = {len(examples)} requests (max_concurrency={MAX_CONCURRENCY})")

results, errors = await evaluate_many(examples, max_concurrency=MAX_CONCURRENCY)
all_results_df = results_to_dataframe(results)

for llm, sub in all_results_df.groupby("llm_name", sort=False):
    sub.to_csv(OUT_DIR / f"pedagogy_eval_{llm}.csv", index=False)

def with_judge(frame):
    out = frame.copy()
    out.insert(0, "judge_model", MODEL)
    return out

damr_by_llm = with_judge(desired_match_rates(all_results_df, ["llm_name"]))
damr_cat = with_judge(desired_match_rates(all_results_df, ["llm_name", "error_category"]))
dist_cat = with_judge(label_distribution(all_results_df, ["llm_name", "error_category"]))
damr_by_llm.to_csv(OUT_DIR / "pedagogy_eval_damr_by_llm.csv", index=False)
damr_cat.to_csv(OUT_DIR / "pedagogy_eval_damr_by_llm_error_category.csv", index=False)
dist_cat.to_csv(OUT_DIR / "pedagogy_eval_label_dist_by_llm_error_category.csv", index=False)

print(f"judge={MODEL}  ok={len(all_results_df)}  fail={len(errors)}")
dims = list(TAXONOMY)
display(damr_by_llm.pivot(index="dimension", columns="llm_name", values="DAMR").reindex(dims))
display(damr_cat.pivot(index=["llm_name", "error_category", "n"], columns="dimension", values="DAMR").reindex(columns=dims))


298 rows × 5 models = 1490 requests (max_concurrency=200)
start: 0/1490 (max_concurrency=200)
progress: 25/1490 (ok 25 / fail 0)
progress: 50/1490 (ok 50 / fail 0)
progress: 75/1490 (ok 75 / fail 0)
progress: 100/1490 (ok 100 / fail 0)
progress: 125/1490 (ok 125 / fail 0)
progress: 150/1490 (ok 150 / fail 0)
progress: 175/1490 (ok 175 / fail 0)
progress: 200/1490 (ok 200 / fail 0)
progress: 225/1490 (ok 225 / fail 0)
progress: 250/1490 (ok 250 / fail 0)
progress: 275/1490 (ok 275 / fail 0)
progress: 300/1490 (ok 300 / fail 0)
progress: 325/1490 (ok 325 / fail 0)
progress: 350/1490 (ok 350 / fail 0)
progress: 375/1490 (ok 375 / fail 0)
progress: 400/1490 (ok 400 / fail 0)
progress: 425/1490 (ok 425 / fail 0)
progress: 450/1490 (ok 450 / fail 0)
progress: 475/1490 (ok 475 / fail 0)
progress: 500/1490 (ok 500 / fail 0)
progress: 525/1490 (ok 525 / fail 0)
progress: 550/1490 (ok 550 / fail 0)
progress: 575/1490 (ok 575 / fail 0)
progress: 600/1490 (ok 600 / fail 0)
progress: 625/1490 (ok 6

llm_name,base,dpo_01,dpo_03,dpo_05,sft
dimension,,,,,
mistake_identification,0.953020,0.630872,0.651007,0.838926,0.416107
mistake_location,0.932886,0.506711,0.513423,0.687919,0.389262
revealing_answer,0.986577,0.701342,0.885906,0.808725,0.939597
providing_guidance,0.885906,0.144295,0.228188,0.271812,0.281879
actionability,0.885906,0.144295,0.271812,0.288591,0.298658
coherence,0.996644,0.909396,0.953020,0.963087,0.976510
tutor_tone,1.000000,0.248322,0.496644,0.348993,0.697987
human_likeness,0.996644,0.483221,0.728188,0.624161,0.845638


dimension                                                    mistake_identification  \
llm_name error_category                                  n                            
base     calculation_error_easily_solved_by_a_calculator 38                0.973684   
         extra_quantity_or_missing_quantity              71                0.943662   
         missing_wrong_factual_knowledge                 42                0.976190   
         misunderstanding_of_a_question                  85                0.952941   
         none_of_the_above                               26                1.000000   
         reached_correct_solution_but_proceeded_further  21                0.904762   
         unit_conversion_error                           15                0.866667   
dpo_01   calculation_error_easily_solved_by_a_calculator 38                0.684211   
         extra_quantity_or_missing_quantity              71                0.591549   
         missing_wrong_factual_knowledge                 42                0.595238   
         misunderstanding_of_a_question                  85                0.623529   
         none_of_the_above                               26                0.807692   
         reached_correct_solution_but_proceeded_further  21                0.619048   
         unit_conversion_error                           15                0.533333   
dpo_03   calculation_error_easily_solved_by_a_calculator 38                0.631579   
         extra_quantity_or_missing_quantity              71                0.577465   
         missing_wrong_factual_knowledge                 42                0.690476   
         misunderstanding_of_a_question                  85                0.647059   
         none_of_the_above                               26                0.730769   
         reached_correct_solution_but_proceeded_further  21                0.761905   
         unit_conversion_error                           15                0.666667   
dpo_05   calculation_error_easily_solved_by_a_calculator 38                0.842105   
         extra_quantity_or_missing_quantity              71                0.746479   
         missing_wrong_factual_knowledge                 42                0.785714   
         misunderstanding_of_a_question                  85                0.870588   
         none_of_the_above                               26                0.923077   
         reached_correct_solution_but_proceeded_further  21                0.952381   
         unit_conversion_error                           15                0.933333   
sft      calculation_error_easily_solved_by_a_calculator 38                0.394737   
         extra_quantity_or_missing_quantity              71                0.352113   
         missing_wrong_factual_knowledge                 42                0.595238   
         misunderstanding_of_a_question                  85                0.376471   
         none_of_the_above                               26                0.346154   
         reached_correct_solution_but_proceeded_further  21                0.476190   
         unit_conversion_error                           15                0.533333   

dimension                                                    mistake_location  \
llm_name error_category                                  n                      
base     calculation_error_easily_solved_by_a_calculator 38          0.973684   
         extra_quantity_or_missing_quantity              71          0.915493   
         missing_wrong_factual_knowledge                 42          0.952381   
         misunderstanding_of_a_question                  85          0.929412   
         none_of_the_above                               26          1.000000   
         reached_correct_solution_but_proceeded_further  21          0.904762   
         unit_conversion_error                           15          0.800000   
dpo_01   calculation_error_easily_solved_by_a_calcu